In [0]:
spark.version


In [0]:
%run ../tests/test_silver_layer

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import logging

In [0]:
# Setup the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("Taxi Silver Layer")

**Databricks Widgets Set Up**

In [0]:
dbutils.widgets.text("bronze_taxi_2025", "")
dbutils.widgets.text("bronze_taxi_2026", "")
dbutils.widgets.text("silver_taxi", "")

bronze_taxi_2025 = dbutils.widgets.get("bronze_taxi_2025") or "chicago_taxi_data.bronze.bronze_taxi_2025"
bronze_taxi_2026 = dbutils.widgets.get("bronze_taxi_2026") or "chicago_taxi_data.bronze.bronze_taxi_2026"
silver_taxi_data = dbutils.widgets.get("silver_taxi") or "chicago_taxi_data.silver.silver_taxi"

In [0]:
df_bronze_taxi_2025 = spark.read.format("delta").table(bronze_taxi_2025)
df_bronze_taxi_2026 = spark.read.format("delta").table(bronze_taxi_2026)


In [0]:
df_bronze_taxi_2025.printSchema()

In [0]:
df_bronze_taxi_2026.printSchema()

**Union Chicago Taxi Data For 2025 and 2026 years**

In [0]:
df_bronze_taxi_2026 = df_bronze_taxi_2026.select(*df_bronze_taxi_2025.columns)
logger.info(f"Chicago Taxi Data 2025 contains: {df_bronze_taxi_2025.count()} records")
logger.info(f"Chicago Taxi Data 2026 contains: {df_bronze_taxi_2026.count()} records")

df_silver_taxi = df_bronze_taxi_2025.unionByName(df_bronze_taxi_2026, allowMissingColumns=True)
total_count = df_silver_taxi.count()
logger.info(f"Unioned Chicago Taxi Data 2025-2026 contains: {total_count}")

**Schema Standardization and Type Casting.**

In [0]:
df_silver_taxi = df_silver_taxi.select(

    "trip_id", 
    "taxi_id",
    
    F.col("trip_start_timestamp"),
    F.to_timestamp(F.col("trip_end_timestamp")).alias("trip_end_timestamp"),
    
    # Financials (Decimal 10,2 is god for USD)
    F.col("fare").cast("decimal(10, 2)"),
    F.col("tips").cast("decimal(10, 2)"),
    F.col("tolls").cast("decimal(10, 2)"),
    F.col("extras").cast("decimal(10, 2)"),
    F.col("trip_total").cast("decimal(10, 2)"),
    
    # Trip Metrics 
    F.col("trip_seconds").cast("int"),
    F.col("trip_miles").cast("double"),
    
    # Geographyfor Analytics
    "payment_type",
    "company",
    F.col("pickup_community_area").cast("int"),
    F.col("dropoff_community_area").cast("int"),
    "pickup_census_tract",
    "dropoff_census_tract",
    
    # Centroids (Cast to Double for Mapping)
    F.col("pickup_centroid_latitude").cast("double"),
    F.col("pickup_centroid_longitude").cast("double"),
    F.col("dropoff_centroid_latitude").cast("double"),
    F.col("dropoff_centroid_longitude").cast("double"),
    
    # Keep the location structs as is for advanced GIS work
    "pickup_centroid_location",
    "dropoff_centroid_location",
    
    # Partition Columns
    "year",
    "month"
)


In [0]:
df_silver_taxi.printSchema()

In [0]:
df_silver_taxi.show(n=1, vertical=True, truncate=False)

**Drop Duplicates**

In [0]:
df_silver_taxi = df_silver_taxi.dropDuplicates()

**Check Missing Values Percentage**

In [0]:
df_silver_taxi.select([F.round((100 * F.sum(F.col(c).isNull().cast("int")) / total_count), 2).alias(f"{c}, %") for c in df_silver_taxi.columns]).show(vertical=True)

**Feature Validation**

Filter out and leave only valid data values

In [0]:
df_silver_taxi = df_silver_taxi.filter(
    (F.col("fare") >= 0.0) & 
    (F.col("trip_miles") >= 0.0) & 
    (F.col("trip_seconds") >= 0.0) &
    (F.col("tips") >= 0.0)
)

clean_count = df_silver_taxi.count()
anomaly_count = total_count - clean_count

logger.info(f"Data Cleaning Summary:")
logger.info(f" - Total records processed: {total_count}")
logger.info(f" - Anomalies removed (negative values): {anomaly_count}")
logger.info(f" - Clean records remaining: {clean_count}")

# Data Quality Alert
if anomaly_count > (total_count * 0.1): # when more than 10% is bad data
    logger.warning(f"High anomaly rate detected: {round((anomaly_count/total_count)*100, 2)}%")

**Feature Engineering**

Handle Nulls  and create additional columns

In [0]:
df_silver_taxi = df_silver_taxi.withColumn(
    "trip_end_timestamp",
    F.when(
        F.col("trip_end_timestamp").isNull() & F.col("trip_start_timestamp").isNotNull(),
        F.col("trip_start_timestamp")
    ).otherwise(F.col("trip_end_timestamp"))
)

# Replace Nulls with 0
df_silver_taxi = df_silver_taxi.withColumn(
    "trip_seconds",
    F.when(F.col("trip_seconds").isNull(), 0
    ).otherwise(F.col("trip_seconds"))
)

cols_to_fill_null = ["fare", "tips", "trip_miles"]

means_dict = df_silver_taxi.select([F.round(F.mean(F.col(c).cast('double')), 2).alias(c) for c in cols_to_fill_null]).first().asDict()

print("Calculated Means:", means_dict)

# Fill missing values with mean of each column
df_silver_taxi = df_silver_taxi.fillna(means_dict)

# Create new columns day, hour, weekday and is_weekend base on trip_start_timestamp
df_silver_taxi = (
    df_silver_taxi.withColumn("date", F.to_date("trip_start_timestamp"))
                .withColumn("day", F.dayofmonth("trip_start_timestamp"))
                .withColumn("hour", F.hour("trip_start_timestamp"))
                .withColumn("week_day", F.date_format("trip_start_timestamp", "E"))
                .withColumn("is_weekend", F.dayofweek("trip_start_timestamp").isin([1, 7]))
)

**Data Quality Check**

In [0]:
required_columns = ["trip_id", "taxi_id", "trip_start_timestamp", "date", "trip_end_timestamp", "fare", "tips", "tolls", "extras", "trip_total", "trip_seconds", "trip_miles", "payment_type", "company", "pickup_community_area"]


try:
    logger.info("Starting Chicago Taxi DQ checks Silver Layer")

    validate_no_nulls(df_silver_taxi, "trip_start_timestamp")
    validate_no_nulls(df_silver_taxi, "trip_id")
    validate_no_nulls(df_silver_taxi, "fare")
    validate_no_nulls(df_silver_taxi, "trip_miles")
    validate_no_nulls(df_silver_taxi, "tips")
    validate_schema(df_silver_taxi, required_columns)
    validate_duplicates(df_silver_taxi, ["trip_start_timestamp", "trip_id"])
    validate_chronology(df_silver_taxi)
    validate_positive_values(df_silver_taxi, ["fare", "trip_miles", "trip_seconds"])
    
    logger.info("Silver Taxi DQ tests Passed.")
except Exception as e:
    logger.error(f"DQ Failed: {str(e)}")
    dbutils.notebook.exit(str(e))

**Save Combined Cleaned Preprocessed Chicago Taxi Data For 2025 and 2026 years to Silver Layer Delta Table**

In [0]:

try:
    logger.info(f"Writing df_silver_taxi to Silver Layer")
    df_silver_taxi.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .partitionBy("year", "month")\
        .saveAsTable(silver_taxi_data)
    logger.info(f"Data written successfully to table `{silver_taxi_data}` Delta table")
except Exception as e:
   logger.error(f"Error writing df_silver_taxi to Silver Layer: {e}")
   dbutils.notebook.exit(e)
